# Job Submission Framework Walkthrough (Maintainer Guide)

This notebook walks through the **job submission feature** introduced in commit:

> `Add job-submission feature`

It is intended for maintainers of `checkmaite` and covers:

- what new modules were added,
- how to use the asynchronous job API end-to-end,
- lifecycle and result semantics,
- analytics-store behavior,
- operational caveats and currently out-of-scope items.

## 1) What changed in this feature

### New jobs subsystem

- `src/checkmaite/jobs/protocol.py`
  - `JobStatus`, `CapabilityRunRef`, job errors, `Job`/`Backend` protocols.
- `src/checkmaite/jobs/_api.py`
  - public orchestration API: `configure_backend`, `submit_capability`, `list_jobs`, `get_job`, `shutdown_backend`.
- `src/checkmaite/jobs/backends/ray/backend.py`
  - Ray Core backend implementation (`RayBackend`, `RayJob`).
- `src/checkmaite/jobs/_store.py`
  - analytics-store construction + run write helpers.

### Analytics store updates used by jobs

- `store_uri` / `get_run_uri(...)` resolve to a **payload object URI** (plain parquet/object path, no fragment convention).
- receipt-based URI resolution ignores the auto-generated `runs` table.
- payload tables remain idempotent by `run_uid`, while the `runs` table remains deduplicated by mapping key `(run_uid, capability_table, entity_type, entity_id)`.

---

**Design intent recap**

- Jobs are **reference-first**: `job.result()` returns `CapabilityRunRef`, not full `CapabilityRunBase` payload.
- `configure_backend(..., analytics_store=...)` requires an explicit client-chosen durable store location.
- Job submission does **not** consult the notebook process' local cache before submission.
- Worker-side analytics store write happens before returning success.
- `store_uri` is a plain payload-object URI.
- `outputs_uri` is currently `None` (full artifact materialization is not part of this version).

In [ ]:
import inspect
import shutil
import socket
import subprocess
import sys
import tempfile
from pathlib import Path

import polars as pl

from checkmaite import cache_path
from checkmaite.core.analytics_store import AnalyticsStore, ParquetBackend
from checkmaite.core.object_detection import DataevalCleaning
from checkmaite.core.object_detection.dataset_loaders import CocoDetectionDataset
from checkmaite.jobs import (
    JobFailedError,
    configure_backend,
    get_job,
    list_jobs,
    shutdown_backend,
    submit_capability,
)

print("submit_capability signature:")
print(inspect.signature(submit_capability))

## 2) Demo setup (isolated cache + isolated analytics store + external Ray cluster)

For maintainers, this notebook intentionally isolates all side effects:

- local cache is redirected to a temporary directory,
- analytics store is explicitly configured on the client (`analytics_store={...}`),
- a **separate Ray cluster process** is started via the Ray CLI (`ray start`),
- the notebook kernel then connects to that cluster via `configure_backend(...)`.

The current backend implementation is intentionally optimized for **single-threaded notebook usage**:
its in-process job registry is simple and does not attempt multi-thread coordination.

In [ ]:
def find_repo_root(start: Path) -> Path:
    target = Path("tests/data_for_tests/coco_dataset/ann_file.json")
    for p in [start, *start.parents]:
        if (p / target).exists():
            return p
    raise FileNotFoundError(
        "Could not find repository root containing tests/data_for_tests/coco_dataset/ann_file.json"
    )


def pick_free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])


def run_ray_cli(*args: str) -> subprocess.CompletedProcess[str]:
    cmd = [sys.executable, "-m", "ray.scripts.scripts", *args]
    print("$", " ".join(cmd))
    completed = subprocess.run(cmd, check=True, capture_output=True, text=True)
    if completed.stdout.strip():
        print(completed.stdout.strip().splitlines()[-1])
    if completed.stderr.strip():
        print(completed.stderr.strip().splitlines()[-1])
    return completed


repo_root = find_repo_root(Path.cwd())
workspace = Path(tempfile.mkdtemp(prefix="checkmaite_jobs_walkthrough_"))
demo_store = workspace / "analytics_store"
demo_cache = workspace / "cache"
demo_store.mkdir(parents=True, exist_ok=True)
demo_cache.mkdir(parents=True, exist_ok=True)

old_cache_path = cache_path()
cache_path(demo_cache)

# Start a separate Ray cluster process (outside notebook kernel)
ray_port = pick_free_port()
ray_client_port = pick_free_port()
ray_address = f"127.0.0.1:{ray_port}"

shutdown_backend(wait=False)
run_ray_cli("stop", "--force")
run_ray_cli(
    "start",
    "--head",
    "--disable-usage-stats",
    "--include-dashboard=False",
    "--port",
    str(ray_port),
    "--ray-client-server-port",
    str(ray_client_port),
)

# Connect the notebook process to the external cluster.
configure_backend(
    "ray",
    address=ray_address,
    analytics_store={"backend": "parquet", "uri": str(demo_store)},
)

print(f"Repo root:      {repo_root}")
print(f"Demo workspace: {workspace}")
print(f"Cache path:     {cache_path()}")
print(f"Store URI:      {demo_store}")
print(f"Ray address:    {ray_address}")

## 3) Build a real checkmaite capability run target

We use a real built-in capability:

- capability: `DataevalCleaning` (object detection)
- dataset: tiny COCO fixture under `tests/data_for_tests/coco_dataset`

This keeps the walkthrough realistic while still lightweight.

In [ ]:
dataset_root = repo_root / "tests" / "data_for_tests" / "coco_dataset"
dataset = CocoDetectionDataset(
    root=str(dataset_root),
    ann_file=str(dataset_root / "ann_file.json"),
    dataset_id="coco-jobs-demo",
)
capability = DataevalCleaning()

print("Dataset:", dataset.metadata["id"], "len=", len(dataset))
print("Capability:", capability.id)

## 4) Submit asynchronously and inspect lifecycle

`submit_capability(...)` mirrors the sync capability API inputs, but returns a `Job[CapabilityRunRef]`.

In [ ]:
job = submit_capability(
    capability,
    datasets=[dataset],
    use_cache=False,
    report_threshold=0.5,
)

print("Job ID:", job.job_id)
print("Initial status snapshot:", job.status)
print("Tracked jobs now:", len(list_jobs()))
print("Lookup by id works:", get_job(job.job_id).job_id == job.job_id)

# Small non-blocking poll
print("Status after short wait:", job.wait(timeout=0.01))

In [ ]:
ref = job.result(timeout=300)

print("Run UID:", ref.run_uid)
print("Capability ID:", ref.capability_id)
print("Store URI:", ref.store_uri)
print("Outputs URI:", ref.outputs_uri)
print("Summary keys:", sorted(ref.summary.keys()))
print("Store URI is a plain payload path:", ref.store_uri.endswith(".parquet") and "#" not in ref.store_uri)

### Current behavior to remember

- `outputs_uri` is expected to be `None`.
- Worker-side full-run artifact materialization/loading is **not included yet**.
- The authoritative durable record is the analytics store write.
- `store_uri` is a payload-object URI, not a `#run_uid=...` decorated reference.

## 5) Read persisted data from analytics store

We can inspect both:

- capability-specific extracted table (`dataeval_cleaning`), and
- auto-generated run history (`runs`).

In [ ]:
store = AnalyticsStore(ParquetBackend(str(demo_store)))
print("Tables:", store.list_tables())
print("Backend lookup matches returned URI:", store.get_run_uri(ref.run_uid) == ref.store_uri)

clean_df = store.query_sql("SELECT * FROM dataeval_cleaning")
print("\ndataeval_cleaning rows:")
print(clean_df.select([c for c in clean_df.columns if c in {"run_uid", "dataset_id", "image_count"}]))

runs_df = store.query_sql(
    "SELECT run_uid, entity_type, entity_id FROM runs"
)
runs_df = runs_df.filter(pl.col("run_uid") == ref.run_uid)
print("\nruns rows for run_uid:")
print(runs_df)

## 6) Local cache is not consulted before submission

Job submission does **not** short-circuit on the notebook process' local run cache.
Even if an equivalent synchronous capability run is already cached locally,
`submit_capability(...)` still submits remote work and the worker executes with
`use_cache=False`.

Because payload-table writes are idempotent by `run_uid`, repeated submissions of
that same run do not create duplicate payload rows in the analytics store.

In [ ]:
# Seed the notebook process' local cache.
seeded_run = capability.run(datasets=[dataset], use_cache=True)
print("Seeded local cache run UID:", seeded_run.run_uid)

job_with_matching_local_cache = submit_capability(capability, datasets=[dataset], use_cache=True)
print("Submitted job type:", type(job_with_matching_local_cache).__name__)

submitted_ref = job_with_matching_local_cache.result(timeout=300)
print("Run UID matches seeded cache:", submitted_ref.run_uid == seeded_run.run_uid)
print("Store URI resolved by backend:", store.get_run_uri(submitted_ref.run_uid) == submitted_ref.store_uri)

payload_rows = store.query_sql(
    f"SELECT run_uid FROM dataeval_cleaning WHERE run_uid = '{submitted_ref.run_uid}'"
)
print("Payload rows for run_uid after repeat submission:", payload_rows.shape[0])

## 7) Store-write failure policy

`RayBackend` now uses a single policy for analytics-store write failures:

- store write failure -> **job failure** (`JobFailedError`)

In other words, submission success implies the analytics-store write step succeeded.

In [ ]:
# Store write failure should always fail the job
bad_store_root = workspace / "not-a-directory"
bad_store_root.write_text("this path is a file, not a directory")

configure_backend(
    "ray",
    address=ray_address,
    force_reinit=True,
    analytics_store={"backend": "parquet", "uri": str(bad_store_root)},
)

failing_job = submit_capability(capability, datasets=[dataset], use_cache=False)
try:
    _ = failing_job.result(timeout=120)
    print("Unexpected: store write failure did not fail the job")
except JobFailedError as exc:
    print("Store write failure raised JobFailedError as expected")
    print(str(exc).split("\n")[0])

## 8) Reconfiguration semantics to remember

- `configure_backend(..., force_reinit=False)` (default):
  - non-blocking handoff intent,
  - does **not** force runtime teardown/reconnect.
- `force_reinit=True`:
  - explicitly disconnects/re-inits Ray,
  - required to guarantee new `address/runtime_env` application,
  - may interrupt in-flight work.
- The in-process backend registry is intentionally simple and currently assumes
  single-threaded notebook-style access.

In [ ]:
# Restore normal backend session for any subsequent work
configure_backend(
    "ray",
    address=ray_address,
    force_reinit=True,
    analytics_store={"backend": "parquet", "uri": str(demo_store)},
)
print("Backend restored to normal mode on external Ray cluster.")

## 9) What is intentionally not included in this feature

For maintainers planning next iterations, this commit **does not** include:

1. Worker-side full artifact persistence + `outputs_uri` population.
2. `load_run(ref)` API for automatic full payload hydration.
3. Registry-based dedupe/reattach durability layer (`05_RAY_REGISTRY.md` scope).
4. Advanced cluster packaging/deployment orchestration beyond Ray runtime_env usage.

Current feature is intentionally focused on:

- stable async job protocol,
- reference-first results,
- analytics-store integration,
- practical Ray backend behavior + tests.

## 10) Cleanup

Run this cell at the end of the walkthrough session.

It disconnects the job backend, restores cache state, and stops the
external Ray cluster process started by this notebook.

In [ ]:
shutdown_backend(wait=False)
cache_path(old_cache_path)

# stop external Ray cluster process
try:
    run_ray_cli("stop", "--force")
except Exception as exc:
    print(f"Ray stop warning: {exc}")

shutil.rmtree(workspace, ignore_errors=True)

print("Cleaned up temporary backend/cache/store state and stopped external Ray cluster.")